# LUPW Flow Traffic Light System — Training Notebook (v4)

## YOLOv8 Object Detection Pipeline

This notebook trains a **YOLOv8 Nano** object detection model to read **rotameter flow meters** by detecting the tube and float positions in camera images.

**Approach:**
1. **YOLOv8 Nano** detects two objects: the rotameter **tube** and the **float** (ball/bobbin)
2. The float's top edge position relative to the tube determines the **flow level**
3. Position ratio maps to a **4-state traffic light** using calibrated thresholds:
   - 🔴 **RED** = No flow (position ≤ zero-pos)
   - 🟡 **AMBER** = Rinse 1 (position ≤ rinse1-pos, ~10 GPM)
   - 🔵 **BLUE** = Rinse 2 (position ≤ rinse2-pos, ~25 GPM)
   - 🟢 **GREEN** = Online (position > rinse2-pos)

**Training environment:** WSL2 Ubuntu with `conda activate vision-ml`
- PyTorch 2.6.0 + CUDA 12.4
- Ultralytics YOLOv8 8.3.x
- Albumentations for augmentation

**Deployment target:** Raspberry Pi 4 with Camera Module 2 and GPIO traffic lights.

## Section 1: Verify Environment & Import Libraries

This notebook runs in the **vision-ml** conda environment. All dependencies should already be installed — no `pip install` needed.

In [ ]:
# Verify all required packages are available
import importlib

required = {
    'ultralytics': 'ultralytics',
    'torch': 'torch',
    'torchvision': 'torchvision',
    'albumentations': 'albumentations',
    'cv2': 'opencv-python',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
    'PIL': 'Pillow',
    'yaml': 'PyYAML',
}

print("Checking vision-ml dependencies:")
for mod, pkg in required.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'ok')
        print(f"  \u2713 {pkg:20s} {ver}")
    except ImportError:
        print(f"  \u2717 {pkg:20s} NOT FOUND (pip install {pkg})")

In [ ]:
import os
import glob
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from pathlib import Path
import yaml

import torch
import torchvision
from ultralytics import YOLO

print(f"PyTorch:      {torch.__version__}")
print(f"Torchvision:  {torchvision.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" ({torch.cuda.get_device_name(0)})")
else:
    print(" (CPU only)")
print(f"Ultralytics:  {importlib.import_module('ultralytics').__version__}")

## Section 2: Dataset Preparation

### YOLO Object Detection Format

YOLOv8 needs **bounding box annotations** — you draw boxes around the **tube** and **float** in each image.

### Required folder structure
```
dataset/
├── data.yaml            ← class names and paths
├── images/
│   ├── train/           ← ~80% of your images
│   │   ├── img_001.jpg
│   │   └── ...
│   └── val/             ← ~20% of your images
│       └── ...
└── labels/
    ├── train/           ← one .txt per image (same filename)
    │   ├── img_001.txt
    │   └── ...
    └── val/
        └── ...
```

### Annotation format (`.txt` label files)
Each line: `class_id  x_center  y_center  width  height`
- All values **normalised** to [0, 1] relative to image dimensions
- Class IDs: `0` = tube, `1` = float

Example `img_001.txt`:
```
0 0.50 0.50 0.15 0.85
1 0.50 0.35 0.08 0.06
```

### Annotation tools (pick one)
- **[Roboflow](https://roboflow.com)** — web-based, free tier, can auto-annotate ★ recommended
- **[Label Studio](https://labelstud.io)** — open source, self-hosted
- **[CVAT](https://www.cvat.ai)** — open source, powerful

### Tips
- Annotate **every image** with both tube and float bounding boxes
- Aim for **200+ images** minimum (500+ recommended)
- Include varied lighting and slight angle changes
- The float position at different flow levels is what the model learns to detect

In [ ]:
# Create dataset directory structure if it doesn't exist
dirs = [
    "dataset/images/train",
    "dataset/images/val",
    "dataset/labels/train",
    "dataset/labels/val",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  \u2713 {d}/")

# Verify data.yaml
if os.path.exists("dataset/data.yaml"):
    with open("dataset/data.yaml") as f:
        config = yaml.safe_load(f)
    print(f"\ndata.yaml loaded:")
    print(f"  Classes: {config['names']}")
    print(f"  nc: {config['nc']}")
    print(f"  Train: {config['train']}")
    print(f"  Val:   {config['val']}")
else:
    print("\n\u26a0 dataset/data.yaml not found \u2014 creating default...")
    config = {
        'path': './dataset',
        'train': 'images/train',
        'val': 'images/val',
        'nc': 2,
        'names': {0: 'tube', 1: 'float'}
    }
    with open("dataset/data.yaml", "w") as f:
        yaml.dump(config, f, default_flow_style=False)
    print("  Created dataset/data.yaml")

In [ ]:
# Dataset statistics and integrity check
def count_dataset():
    splits = {'train': 0, 'val': 0}
    label_splits = {'train': 0, 'val': 0}

    for split in splits:
        img_dir = f"dataset/images/{split}"
        lbl_dir = f"dataset/labels/{split}"

        if os.path.exists(img_dir):
            imgs = [f for f in os.listdir(img_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            splits[split] = len(imgs)

        if os.path.exists(lbl_dir):
            lbls = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
            label_splits[split] = len(lbls)

    print("Dataset Summary:")
    print(f"  Train: {splits['train']} images, {label_splits['train']} labels")
    print(f"  Val:   {splits['val']} images, {label_splits['val']} labels")
    print(f"  Total: {splits['train'] + splits['val']} images")

    if splits['train'] == 0:
        print("\n\u26a0 No training images found!")
        print("  Add annotated images to dataset/images/train/")
        print("  and corresponding labels to dataset/labels/train/")
        return

    # Check for images without labels
    for split in ['train', 'val']:
        img_dir = f"dataset/images/{split}"
        lbl_dir = f"dataset/labels/{split}"
        if os.path.exists(img_dir) and os.path.exists(lbl_dir):
            imgs = {Path(f).stem for f in os.listdir(img_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))}
            lbls = {Path(f).stem for f in os.listdir(lbl_dir) if f.endswith('.txt')}
            missing = imgs - lbls
            if missing:
                print(f"\n\u26a0 {split}: {len(missing)} images without labels: {list(missing)[:5]}...")
            extra = lbls - imgs
            if extra:
                print(f"\n\u26a0 {split}: {len(extra)} labels without images: {list(extra)[:5]}...")

count_dataset()

## Section 3: Visualize Dataset with Annotations

Display sample images with their bounding box annotations overlaid, and analyse the label class distribution.

In [ ]:
def visualize_annotations(split="train", n=8):
    """Show sample images with bounding box annotations."""
    img_dir = f"dataset/images/{split}"
    lbl_dir = f"dataset/labels/{split}"

    if not os.path.exists(img_dir):
        print(f"No {split} images found at {img_dir}/")
        return

    img_files = sorted([f for f in os.listdir(img_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not img_files:
        print(f"No images in {img_dir}/")
        return

    n = min(n, len(img_files))
    sample = random.sample(img_files, n) if len(img_files) >= n else img_files

    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = axes.flat

    class_names = {0: "tube", 1: "float"}
    class_colors = {0: "cyan", 1: "lime"}

    for ax, img_file in zip(axes, sample):
        img_path = os.path.join(img_dir, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        ax.imshow(img)

        # Load labels
        lbl_file = os.path.join(lbl_dir, Path(img_file).stem + ".txt")
        if os.path.exists(lbl_file):
            with open(lbl_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id = int(parts[0])
                        cx, cy, bw, bh = [float(x) for x in parts[1:]]
                        x1 = (cx - bw / 2) * w
                        y1 = (cy - bh / 2) * h
                        box_w, box_h = bw * w, bh * h
                        color = class_colors.get(cls_id, "red")
                        rect = patches.Rectangle(
                            (x1, y1), box_w, box_h,
                            linewidth=2, edgecolor=color, facecolor='none')
                        ax.add_patch(rect)
                        ax.text(x1, y1 - 5, class_names.get(cls_id, str(cls_id)),
                                color=color, fontsize=9, fontweight='bold',
                                bbox=dict(boxstyle='round,pad=0.2',
                                          facecolor='black', alpha=0.7))
        ax.set_title(img_file, fontsize=9)
        ax.axis("off")

    for ax in list(axes)[n:]:
        ax.axis("off")

    plt.suptitle(f"Annotated {split.title()} Samples", fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_annotations("train", n=8)

In [ ]:
def analyze_labels(split="train"):
    """Analyse label distribution and bounding box statistics."""
    lbl_dir = f"dataset/labels/{split}"
    if not os.path.exists(lbl_dir):
        print(f"No labels in {lbl_dir}/")
        return

    class_counts = {}
    widths = {}
    heights = {}

    lbl_files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
    for lbl_file in lbl_files:
        with open(os.path.join(lbl_dir, lbl_file)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    bw, bh = float(parts[3]), float(parts[4])
                    class_counts[cls_id] = class_counts.get(cls_id, 0) + 1
                    widths.setdefault(cls_id, []).append(bw)
                    heights.setdefault(cls_id, []).append(bh)

    if not class_counts:
        print("No labels found.")
        return

    class_names = {0: "tube", 1: "float"}

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Class distribution
    names = [class_names.get(k, str(k)) for k in sorted(class_counts.keys())]
    counts = [class_counts[k] for k in sorted(class_counts.keys())]
    axes[0].bar(names, counts, color=["cyan", "lime"], edgecolor="black")
    axes[0].set_title(f"Class Distribution ({split})")
    axes[0].set_ylabel("Count")

    # Box size scatter
    for cls_id in sorted(widths.keys()):
        if widths[cls_id]:
            axes[1].scatter(widths[cls_id], heights[cls_id],
                            alpha=0.5, label=class_names.get(cls_id, str(cls_id)), s=20)
    axes[1].set_xlabel("Box Width (normalised)")
    axes[1].set_ylabel("Box Height (normalised)")
    axes[1].set_title("Bounding Box Sizes")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"\n{split.title()} Label Statistics:")
    for cls_id in sorted(class_counts.keys()):
        name = class_names.get(cls_id, str(cls_id))
        n = class_counts[cls_id]
        print(f"  {name}: {n} instances")

analyze_labels("train")

## Section 4: Train YOLOv8 Model

Train a **YOLOv8 Nano** model on the rotameter dataset. YOLOv8n is the smallest and fastest variant — ideal for Raspberry Pi deployment.

**Training config:**
- Base model: `yolov8n.pt` (pretrained on COCO, fine-tuned on our data)
- Image size: 640×640
- Epochs: 100 (with early stopping, patience=20)
- Built-in augmentation: HSV jitter, rotation ±5°, translation, scale, mosaic

The trained weights are saved to `runs/detect/rotameter/weights/best.pt`.

In [ ]:
# \u2500\u2500 Train YOLOv8 Nano on rotameter dataset \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
model = YOLO("yolov8n.pt")  # Load pretrained YOLOv8 Nano

results = model.train(
    data="dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,             # Early stopping
    name="rotameter",
    project="runs/detect",
    device=0 if torch.cuda.is_available() else "cpu",
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5.0,             # Rotation \u00b15\u00b0
    translate=0.1,           # Translation \u00b110%
    scale=0.3,               # Scale \u00b130%
    flipud=0.0,              # No vertical flip (fixed orientation)
    fliplr=0.0,              # No horizontal flip
    mosaic=0.5,
    close_mosaic=10,         # Disable mosaic for last 10 epochs
    workers=4,
    verbose=True,
)

print("\n\u2713 Training complete!")
print(f"  Best weights: runs/detect/rotameter/weights/best.pt")

In [ ]:
# \u2500\u2500 Plot training results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
results_csv = "runs/detect/rotameter/results.csv"

if os.path.exists(results_csv):
    df_results = pd.read_csv(results_csv)
    df_results.columns = df_results.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("YOLOv8 Training Results", fontsize=14)

    metrics = [
        ("train/box_loss", "Train Box Loss"),
        ("train/cls_loss", "Train Class Loss"),
        ("train/dfl_loss", "Train DFL Loss"),
        ("metrics/precision(B)", "Precision"),
        ("metrics/recall(B)", "Recall"),
        ("metrics/mAP50(B)", "mAP@50"),
    ]

    for ax, (col, title) in zip(axes.flat, metrics):
        if col in df_results.columns:
            ax.plot(df_results[col], linewidth=2)
            ax.set_title(title)
            ax.set_xlabel("Epoch")
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150)
    plt.show()

    # Print final metrics
    last = df_results.iloc[-1]
    print(f"\nFinal Metrics (Epoch {len(df_results)}):")
    for col in ["metrics/precision(B)", "metrics/recall(B)",
                "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        if col in df_results.columns:
            print(f"  {col.split('/')[-1]}: {last[col]:.4f}")
else:
    print("No results.csv found \u2014 run training first (Section 4).")

## Section 5: Evaluate Model Performance

Run validation on the held-out val set and analyse:
- mAP (mean Average Precision) at IoU 50% and 50–95%
- Per-class precision, recall, and AP
- Visual predictions on sample images

In [ ]:
# \u2500\u2500 Validate the best model \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
best_model = YOLO("runs/detect/rotameter/weights/best.pt")

val_results = best_model.val(
    data="dataset/data.yaml",
    imgsz=640,
    batch=16,
    verbose=True,
)

print(f"\nValidation Results:")
print(f"  mAP@50:     {val_results.box.map50:.4f}")
print(f"  mAP@50-95:  {val_results.box.map:.4f}")
print(f"  Precision:  {val_results.box.mp:.4f}")
print(f"  Recall:     {val_results.box.mr:.4f}")

# Per-class metrics
class_names = {0: "tube", 1: "float"}
for i, (p, r, ap50, ap) in enumerate(zip(
        val_results.box.p, val_results.box.r,
        val_results.box.ap50, val_results.box.ap)):
    print(f"  {class_names.get(i, i):6s}: P={p:.3f}, R={r:.3f}, "
          f"AP50={ap50:.3f}, AP50-95={ap:.3f}")

In [ ]:
# \u2500\u2500 Visual predictions on validation images \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
val_img_dir = "dataset/images/val"

if os.path.exists(val_img_dir):
    val_images = sorted([os.path.join(val_img_dir, f)
                         for f in os.listdir(val_img_dir)
                         if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    n_show = min(8, len(val_images))
    if n_show > 0:
        sample_imgs = random.sample(val_images, n_show)
        cols = min(4, n_show)
        rows = (n_show + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
        if n_show == 1:
            axes = np.array([axes])
        axes = axes.flat

        for ax, img_path in zip(axes, sample_imgs):
            results = best_model(img_path, verbose=False)
            annotated = results[0].plot()
            annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            ax.imshow(annotated)
            ax.set_title(Path(img_path).name, fontsize=9)
            ax.axis("off")

        for ax in list(axes)[n_show:]:
            ax.axis("off")

        plt.suptitle("Model Predictions on Validation Images", fontsize=14)
        plt.tight_layout()
        plt.savefig("val_predictions.png", dpi=150)
        plt.show()
    else:
        print("No validation images found.")
else:
    print(f"Val directory not found: {val_img_dir}")

## Section 6: 4-State Traffic Light Logic (Position-Based Calibration)

The flow level is determined by the **float's top edge position** relative to the **tube**:
- Float at bottom → position ratio ≈ 0 → no flow
- Float at top → position ratio ≈ 1 → max flow

The position ratio maps to a **4-state traffic light** using calibrated thresholds:

| State | Condition | LED |
|---|---|---|
| 🔴 **RED** | Position ≤ zero-pos | Red on |
| 🟡 **AMBER** | Position ≤ rinse1-pos (~10 GPM) | Amber on |
| 🔵 **BLUE** | Position ≤ rinse2-pos (~25 GPM) | Blue on |
| 🟢 **GREEN** | Position > rinse2-pos | Green on |

**GPIO Pins:** GREEN = GPIO 17 (Pin 11), AMBER = GPIO 22 (Pin 15), BLUE = GPIO 23 (Pin 16), RED = GPIO 27 (Pin 13)

**Why position-based?** Rotameter scales are non-linear (wider spacing at low flow, compressed at high flow). Instead of calculating GPM (which would be inaccurate), we compare the float position directly against calibrated thresholds. Calibrate once per rotameter type by measuring where the 10 GPM and 25 GPM marks fall as a ratio of the total tube height.

In [ ]:
# ── Position calculation from detections ──────────────────────────
def calculate_position_from_detections(results, conf_threshold=0.5):
    """
    Calculate float position ratio from YOLOv8 detection results.

    Detects 'tube' (class 0) and 'float' (class 1), then computes
    the float's top edge position as a ratio of the tube height.

    Returns: position_ratio (0.0–1.0) or None if detection fails.
    """
    CLASS_TUBE, CLASS_FLOAT = 0, 1
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return None

    tube_box = float_box = None
    for box in boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        if conf < conf_threshold:
            continue
        xyxy = box.xyxy[0].cpu().numpy()
        if cls == CLASS_TUBE and (tube_box is None or conf > tube_box[1]):
            tube_box = (xyxy, conf)
        elif cls == CLASS_FLOAT and (float_box is None or conf > float_box[1]):
            float_box = (xyxy, conf)

    if tube_box is None or float_box is None:
        return None

    tube_top = tube_box[0][1]
    tube_bottom = tube_box[0][3]
    float_top_y = float_box[0][1]  # Top edge of float = reading point
    tube_height = tube_bottom - tube_top

    if tube_height <= 0:
        return None

    # Higher in image = smaller y = more flow
    position_ratio = max(0.0, min(1.0, (tube_bottom - float_top_y) / tube_height))
    return position_ratio


# ── 4-State traffic light decision (position-based) ──────────────
# Calibration defaults — adjust per rotameter type
ZERO_POS = 0.05     # Below this = no flow (RED)
RINSE1_POS = 0.22   # Below this = rinse 1 (AMBER, ~10 GPM)
RINSE2_POS = 0.45   # Below this = rinse 2 (BLUE, ~25 GPM)

def get_traffic_state(position_ratio, zero_pos=ZERO_POS,
                      rinse1_pos=RINSE1_POS, rinse2_pos=RINSE2_POS):
    """
    Map float position ratio to 4-state traffic light.
      position ≤ zero_pos   → RED   (no flow)
      position ≤ rinse1_pos → AMBER (rinse 1, ~10 GPM)
      position ≤ rinse2_pos → BLUE  (rinse 2, ~25 GPM)
      position > rinse2_pos → GREEN (online)
    """
    if position_ratio <= zero_pos:
        return "RED"
    elif position_ratio <= rinse1_pos:
        return "AMBER"
    elif position_ratio <= rinse2_pos:
        return "BLUE"
    else:
        return "GREEN"


# ── Test the logic ───────────────────────────────────────────────
test_positions = [
    (0.00,  "Float at very bottom"),
    (0.05,  "At zero-pos threshold"),
    (0.10,  "Low flow"),
    (0.22,  "At rinse1-pos threshold (~10 GPM)"),
    (0.30,  "Between rinse 1 and rinse 2"),
    (0.45,  "At rinse2-pos threshold (~25 GPM)"),
    (0.60,  "Online flow"),
    (0.90,  "High online flow"),
]

state_symbols = {"RED": "\U0001f534", "AMBER": "\U0001f7e1", "BLUE": "\U0001f535", "GREEN": "\U0001f7e2"}

print("4-State Traffic Light Logic Test Cases:")
print(f"  Thresholds: zero={ZERO_POS}, rinse1={RINSE1_POS}, rinse2={RINSE2_POS}")
print("-" * 65)
for pos, desc in test_positions:
    state = get_traffic_state(pos)
    print(f"  {pos:5.2f} ({pos*100:5.1f}%) → {state_symbols[state]} {state:6s}  ({desc})")

## Section 7: Export Model for Raspberry Pi

Export the trained model for deployment on the Raspberry Pi 4.

**Option 1: PyTorch (`.pt`)** — simplest, requires `ultralytics` on the Pi \
**Option 2: TorchScript (`.torchscript`)** — no ultralytics needed, just PyTorch \
**Option 3: ONNX (`.onnx`)** — cross-platform (requires `pip install onnx onnxruntime`)

We use **Option 1 (.pt)** as default since `requirements_pi.txt` already includes ultralytics.

In [ ]:
# ── Export model ─────────────────────────────────────────────────
best_model = YOLO("runs/detect/rotameter/weights/best.pt")

# PyTorch model size
pt_path = "runs/detect/rotameter/weights/best.pt"
pt_size = os.path.getsize(pt_path) / (1024 * 1024)
print(f"PyTorch model size:  {pt_size:.2f} MB")

# Export to TorchScript (optional)
try:
    best_model.export(format="torchscript", imgsz=640)
    ts_path = "runs/detect/rotameter/weights/best.torchscript"
    if os.path.exists(ts_path):
        ts_size = os.path.getsize(ts_path) / (1024 * 1024)
        print(f"TorchScript model:   {ts_size:.2f} MB")
except Exception as e:
    print(f"TorchScript export skipped: {e}")

# Export to ONNX (optional)
try:
    best_model.export(format="onnx", imgsz=640, simplify=True)
    onnx_path = "runs/detect/rotameter/weights/best.onnx"
    if os.path.exists(onnx_path):
        onnx_size = os.path.getsize(onnx_path) / (1024 * 1024)
        print(f"ONNX model:          {onnx_size:.2f} MB")
except Exception as e:
    print(f"ONNX export skipped (pip install onnx to enable): {e}")

print(f"\n✓ Deploy to Pi:")
print(f"  scp {pt_path} pi@<PI_IP>:~/lupw-project/best.pt")
print(f"  Then run: python3 inference.py --model best.pt --calibrate")
print(f"  Or with thresholds: python3 inference.py --model best.pt --zero-pos 0.05 --rinse1-pos 0.22 --rinse2-pos 0.45")

## Section 8: Raspberry Pi Hardware Test Scripts

Generate test scripts to verify GPIO wiring and camera setup on the Pi before running inference.

See `setup_guide.md` for full wiring diagram with transistor driver circuits for 5m cable runs (4 LEDs: red, amber, blue, green).

In [ ]:
# ── Generate GPIO test script (4-state) ──────────────────────────
gpio_test_script = '''#!/usr/bin/env python3
"""Test script: Cycle through RED -> AMBER -> BLUE -> GREEN to verify 4-LED wiring."""
import time
import RPi.GPIO as GPIO

GREEN_PIN = 17   # Pin 11
AMBER_PIN = 22   # Pin 15
BLUE_PIN = 23    # Pin 16
RED_PIN = 27     # Pin 13

GPIO.setmode(GPIO.BCM)
GPIO.setwarnings(False)
GPIO.setup(GREEN_PIN, GPIO.OUT)
GPIO.setup(AMBER_PIN, GPIO.OUT)
GPIO.setup(BLUE_PIN, GPIO.OUT)
GPIO.setup(RED_PIN, GPIO.OUT)

def all_off():
    GPIO.output(GREEN_PIN, GPIO.LOW)
    GPIO.output(AMBER_PIN, GPIO.LOW)
    GPIO.output(BLUE_PIN, GPIO.LOW)
    GPIO.output(RED_PIN, GPIO.LOW)

print("Cycling traffic light LEDs... Press Ctrl+C to stop.")
try:
    for i in range(5):
        all_off()
        GPIO.output(RED_PIN, GPIO.HIGH)
        print(f"  [{i+1}] RED on")
        time.sleep(1.5)

        all_off()
        GPIO.output(AMBER_PIN, GPIO.HIGH)
        print(f"  [{i+1}] AMBER on")
        time.sleep(1.5)

        all_off()
        GPIO.output(BLUE_PIN, GPIO.HIGH)
        print(f"  [{i+1}] BLUE on")
        time.sleep(1.5)

        all_off()
        GPIO.output(GREEN_PIN, GPIO.HIGH)
        print(f"  [{i+1}] GREEN on")
        time.sleep(1.5)
    print("Test complete! All 4 LEDs verified.")
except KeyboardInterrupt:
    print("\\nStopped.")
finally:
    all_off()
    GPIO.cleanup()
'''

camera_test_script = '''#!/usr/bin/env python3
"""Test script: Capture a single image from Pi Camera 2."""
import time
from picamera2 import Picamera2

cam = Picamera2()
config = cam.create_preview_configuration(main={"size": (640, 480), "format": "RGB888"})
cam.configure(config)
cam.start()
time.sleep(2)
cam.capture_file("camera_test.jpg")
cam.stop()
print("Saved camera_test.jpg \u2014 check that the rotameter is visible and in frame.")
'''

with open("test_gpio.py", "w") as f:
    f.write(gpio_test_script)
with open("test_camera.py", "w") as f:
    f.write(camera_test_script)

print("Generated test scripts for the Raspberry Pi:")
print("  test_gpio.py   \u2014 cycles RED \u2192 AMBER \u2192 BLUE \u2192 GREEN to verify 4-LED wiring")
print("  test_camera.py \u2014 captures one image to verify camera setup")
print()
print("Transfer to Pi and run:")
print("  scp test_gpio.py test_camera.py pi@<PI_IP>:~/lupw-project/")
print("  ssh pi@<PI_IP>")
print("  cd ~/lupw-project && python3 test_gpio.py")

## Section 9: End-to-End Simulation

Simulate the full pipeline with synthetic position readings to demonstrate the 4-state traffic light behaviour, including debounce logic.

In [ ]:
# ── End-to-End Simulation ─────────────────────────────────────────
from datetime import datetime

ZERO_POS = 0.05
RINSE1_POS = 0.22
RINSE2_POS = 0.45
DEBOUNCE = 3

# Simulate position scenario: online -> rinse2 -> rinse1 -> zero -> recovery -> online
np.random.seed(42)
sim_positions = np.concatenate([
    np.random.uniform(0.5, 0.9, 15),                # Online
    np.linspace(0.7, 0.30, 8),                       # Dropping toward rinse 2
    np.random.uniform(0.23, 0.44, 10),               # Rinse 2 zone
    np.linspace(0.35, 0.10, 8),                      # Dropping toward rinse 1
    np.random.uniform(0.06, 0.21, 8),                # Rinse 1 zone
    np.linspace(0.15, 0.02, 5),                      # Dropping to zero
    np.random.uniform(0.0, 0.04, 6),                 # Zero flow
    np.linspace(0.02, 0.55, 8),                      # Recovery
    np.random.uniform(0.5, 0.85, 8),                 # Back online
])
sim_positions = np.clip(sim_positions, 0, 1.0)

# Apply traffic light logic with debounce
sim_states = []
current_state = "RED"
pending_state = "RED"
debounce_counter = 0

for pos in sim_positions:
    new_state = get_traffic_state(pos, ZERO_POS, RINSE1_POS, RINSE2_POS)
    if new_state == pending_state:
        debounce_counter += 1
    else:
        pending_state = new_state
        debounce_counter = 1
    if debounce_counter >= DEBOUNCE:
        current_state = pending_state
    sim_states.append(current_state)

# ── Plot ────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8),
                                gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle("End-to-End Simulation \u2014 4-State Traffic Light", fontsize=14)

timesteps = range(len(sim_positions))

# Position chart
ax1.plot(timesteps, sim_positions, "b-o", markersize=4, label="Float Position Ratio")
ax1.axhline(y=ZERO_POS, color="red", linestyle="--", linewidth=1.5,
            label=f"Zero threshold ({ZERO_POS})")
ax1.axhline(y=RINSE1_POS, color="orange", linestyle="--", linewidth=1.5,
            label=f"Rinse 1 threshold ({RINSE1_POS})")
ax1.axhline(y=RINSE2_POS, color="blue", linestyle="--", linewidth=1.5,
            label=f"Rinse 2 threshold ({RINSE2_POS})")
ax1.fill_between(timesteps, 0, ZERO_POS, alpha=0.05, color="red")
ax1.fill_between(timesteps, ZERO_POS, RINSE1_POS, alpha=0.05, color="orange")
ax1.fill_between(timesteps, RINSE1_POS, RINSE2_POS, alpha=0.05, color="blue")
ax1.set_ylabel("Position Ratio")
ax1.set_title("Float Position Over Time")
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

# Traffic light state bar
state_colors = {"RED": "red", "AMBER": "#FFA500", "BLUE": "#0066FF", "GREEN": "green"}
colors = [state_colors[s] for s in sim_states]
ax2.bar(timesteps, [1] * len(sim_states), color=colors, edgecolor="none")
ax2.set_ylabel("Light")
ax2.set_xlabel("Reading #")
ax2.set_yticks([])
ax2.set_title("Traffic Light State")

plt.tight_layout()
plt.savefig("demo_simulation.png", dpi=150)
plt.show()

# Summary
print(f"\nSimulation Summary ({len(sim_positions)} readings):")
print(f"  RED   (no flow):    {sim_states.count('RED')}")
print(f"  AMBER (rinse 1):    {sim_states.count('AMBER')}")
print(f"  BLUE  (rinse 2):    {sim_states.count('BLUE')}")
print(f"  GREEN (online):     {sim_states.count('GREEN')}")

## Next Steps

1. **Collect video** — record the rotameter with `record_video.py` on the Pi
2. **Extract frames** — run `extract_frames.py` on your workstation
3. **Annotate** — use [Roboflow](https://roboflow.com) to draw bounding boxes around the **tube** (full tube) and **float** (main body only, exclude top extension)
4. **Organise** — export in YOLO format into `dataset/images/` and `dataset/labels/`
5. **Train** — run Section 4 (uses your vision-ml GPU environment)
6. **Transfer model** — copy `runs/detect/rotameter/weights/best.pt` to the Pi
7. **Wire hardware** — follow `setup_guide.md` for 4-LED wiring with transistor drivers
8. **Test** — run `test_gpio.py` and `test_camera.py` on the Pi
9. **Calibrate** — run `python3 inference.py --model best.pt --calibrate` to measure position ratios
10. **Run inference** — `python3 inference.py --model best.pt --zero-pos 0.05 --rinse1-pos 0.22 --rinse2-pos 0.45`

### Key Parameters

| Parameter | Where | Description |
|---|---|---|
| `--zero-pos` | inference.py | Position ratio below which = no flow / RED (default: 0.05) |
| `--rinse1-pos` | inference.py | Position ratio for rinse 1 / AMBER threshold (default: 0.22) |
| `--rinse2-pos` | inference.py | Position ratio for rinse 2 / BLUE threshold (default: 0.45) |
| `--confidence` | inference.py | Detection confidence threshold (default: 0.5) |
| `--interval` | inference.py | Seconds between readings (default: 0.5) |
| `--calibrate` | inference.py | Run in calibration mode (prints live position ratio) |
| `epochs` | Training notebook | Training epochs (default: 100) |
| `imgsz` | Training notebook | Image size for training (default: 640) |